# Tutorial de IA responsable con Fairlearn y Aequitas

Este notebook muestra un flujo completo para **entrenar, evaluar y auditar un modelo de clasificación** sobre el conjunto de datos COMPAS.

Se utilizan dos bibliotecas complementarias:

- **Fairlearn**: aplica técnicas de preprocesamiento para reducir la dependencia entre los datos y los atributos sensibles.
- **Aequitas**: calcula métricas por grupo, disparidades respecto a grupos de referencia y evaluaciones de equidad.

<!-- Al terminar el tutorial se habrán realizado estas etapas:

1. Carga y filtrado de los datos.
2. Preparación de características y contexto de auditoría.
3. Mitigación opcional mediante Fairlearn.
4. Entrenamiento con validación cruzada estratificada.
5. Evaluación del rendimiento predictivo.
6. Auditoría de disparidades y equidad con Aequitas. -->

## Estructura del flujo

El proyecto está dividido en tres módulos:

- `preprocessing.py`: carga los datos, aplica los filtros, codifica las variables y ejecuta el método de Fairlearn elegido.
- `train_model.py`: entrena el modelo mediante validación cruzada y genera predicciones fuera de muestra.
- `auditar.py`: entrega esas predicciones a Aequitas y calcula los resultados de equidad.

Separar el proceso en módulos permite reutilizar las funciones, probar distintos modelos y comparar métodos de mitigación sin duplicar código.

In [2]:
from pathlib import Path
import warnings

import pandas as pd
from IPython.display import display

from preprocessing import Preprocesar
from train_model import train_model
from auditar import auditar_modelo

warnings.filterwarnings("ignore")

## Conjunto de datos

El archivo utilizado es `compas-scores-two-years.csv`, que contiene información demográfica, antecedentes, datos del cargo y la variable de reincidencia a dos años.

La variable objetivo es `two_year_recid`, que indica si el individuo reincidió en un plazo de dos años. Se trata de un problema de clasificación binaria, donde 1 significa reincidencia y 0 significa no reincidencia.

In [3]:
csv_path = Path("data") / "raw" / "compas-scores-two-years.csv"

print(f"Conjunto de datos: {csv_path.resolve()}")
display(pd.read_csv(csv_path).head())

Conjunto de datos: D:\Escuela\Maestría\Proyecto Tecnologico\Tutorial\data\raw\compas-scores-two-years.csv


,id,name,first,last,compas_screening_date,sex,dob,age,age_cat,race,...,v_decile_score,v_score_text,v_screening_date,in_custody,out_custody,priors_count.1,start,end,event,two_year_recid
0,1,miguel hernandez,miguel,hernandez,2013-08-14,Male,1947-04-18,69,Greater than 45,Other,...,1,Low,2013-08-14,2014-07-07,2014-07-14,0,0,327,0,0
1,3,kevon dixon,kevon,dixon,2013-01-27,Male,1982-01-22,34,25 - 45,African-American,...,1,Low,2013-01-27,2013-01-26,2013-02-05,0,9,159,1,1
2,4,ed philo,ed,philo,2013-04-14,Male,1991-05-14,24,Less than 25,African-American,...,3,Low,2013-04-14,2013-06-16,2013-06-16,4,0,63,0,1
3,5,marcu brown,marcu,brown,2013-01-13,Male,1993-01-21,23,Less than 25,African-American,...,6,Medium,2013-01-13,NaN,NaN,1,0,1174,0,0
4,6,bouthy pierrelouis,bouthy,pierrelouis,2013-03-26,Male,1973-01-22,43,25 - 45,Other,...,1,Low,2013-03-26,NaN,NaN,2,0,1102,0,0


## Preprocesamiento y mitigación con Fairlearn

La función `Preprocesar` realiza las siguientes operaciones:

1. Lee el archivo CSV.
2. Aplica filtros para conservar observaciones válidas del análisis de COMPAS.
3. Selecciona las variables usadas por el modelo.
4. Conserva por separado `race`, `sex` y `age_cat` con sus valores originales para la auditoría.
5. Imputa valores numéricos ausentes con la mediana.
6. Codifica las variables categóricas con `OrdinalEncoder`.
7. Aplica opcionalmente un método de Fairlearn.

### Métodos disponibles

- `None`: no aplica mitigación; sirve como modelo base.
- `"CorrelationRemover"`: transforma las variables no sensibles para retirar la correlación lineal con `race` y `sex`.
- `"PrototypeRepresentationLearner"`: aprende una nueva representación basada en prototipos para conservar utilidad predictiva y reducir diferencias entre grupos.

El preprocesamiento produce dos tablas:

- `compas_preprocessed`: variables numéricas listas para entrenar y la etiqueta `two_year_recid`.
- `compas_audit_context`: identificador, atributos protegidos y etiqueta real sin codificar.

In [4]:
# Cambia este valor para comparar diferentes estrategias de mitigación.
PREPROCESSING_METHOD = "PrototypeRepresentationLearner"
# Alternativas:
# PREPROCESSING_METHOD = "CorrelationRemover"
# PREPROCESSING_METHOD = None

compas_preprocessed, compas_audit_context = Preprocesar(
    csv_path,
    preprocessing_method=PREPROCESSING_METHOD,
)

### Inspección de las salidas

Es importante comprobar que ambas tablas tengan el mismo número de filas. La correspondencia fila a fila permite unir correctamente cada predicción con sus atributos protegidos durante la auditoría.

Las columnas del conjunto procesado pueden cambiar según el método:

- Sin mitigación se conservan las características codificadas.
- `CorrelationRemover` elimina las columnas sensibles del resultado transformado.
- `PrototypeRepresentationLearner` genera columnas nuevas llamadas `prototype_0`, `prototype_1`, etc.

In [5]:
print("Método de preprocesamiento:", PREPROCESSING_METHOD)
print("Forma del conjunto para entrenamiento:", compas_preprocessed.shape)
print("Forma del contexto de auditoría:", compas_audit_context.shape)

print("\nDatos preparados para entrenamiento:")
display(compas_preprocessed.head())

print("\nContexto conservado para la auditoría:")
display(compas_audit_context.head())

Método de preprocesamiento: PrototypeRepresentationLearner
Forma del conjunto para entrenamiento: (6172, 3)
Forma del contexto de auditoría: (6172, 5)

Datos preparados para entrenamiento:


,prototype_0,prototype_1,two_year_recid
0,0.222960,0.777040,0
1,0.674250,0.325750,1
2,0.832128,0.167872,1
3,0.492094,0.507906,0
4,0.473768,0.526232,1



Contexto conservado para la auditoría:


,entity_id,race,sex,age_cat,label_value
0,1,Other,Male,Greater than 45,0
1,3,African-American,Male,25 - 45,1
2,4,African-American,Male,Less than 25,1
3,7,Other,Male,25 - 45,0
4,8,Caucasian,Male,25 - 45,1


## Entrenamiento con validación cruzada

La función `train_model` usa por defecto una **regresión logística** y una validación cruzada `StratifiedKFold` de 10 particiones.

La validación estratificada conserva aproximadamente la proporción de las clases en cada fold. En cada iteración:

1. Se entrena un modelo con nueve particiones.
2. Se predice la partición restante.
3. Se calculan `accuracy`, `precision`, `recall` y `F1`.
4. Se guarda el contexto de las observaciones de prueba junto con sus predicciones.

Al finalizar, cada persona tiene una predicción obtenida por un modelo que **no fue entrenado con esa misma observación**. Esto evita auditar predicciones hechas directamente sobre los datos de entrenamiento.

In [6]:
result, audit_predictions = train_model(
    compas_preprocessed,
    compas_audit_context,
    model_name=f"LR_{PREPROCESSING_METHOD or 'sin_mitigacion'}",
    feature_set=PREPROCESSING_METHOD or "sin_mitigacion",
    n_splits=10,
    random_state=42,
)

### Métricas predictivas

El resultado incluye:

- `metrics`: promedio de cada métrica entre los folds.
- `metrics_std`: desviación estándar entre folds.
- `fold_metrics`: resultados individuales de cada partición.

Estas métricas miden la utilidad predictiva general, pero **no indican por sí solas si el modelo se comporta de forma similar entre grupos**. Esa parte se analiza después con Aequitas.

In [7]:
metrics_summary = pd.DataFrame(
    [
        result["metrics"],
        result["metrics_std"],
    ],
    index=[
        "Media",
        "Desviación estándar",
    ],
)

fold_metrics = pd.DataFrame(result["fold_metrics"]).set_index("fold")

print("Resumen de rendimiento:")
display(metrics_summary)

print("\nMétricas por fold:")
display(fold_metrics)

Resumen de rendimiento:


,accuracy,precision,recall,f1_score
Media,0.575662,0.537022,0.495200,0.514926
Desviación estándar,0.015027,0.018288,0.025027,0.017635



Métricas por fold:


,accuracy,precision,recall,f1_score
fold,,,,
1,0.587379,0.553279,0.480427,0.514286
2,0.577670,0.541322,0.466192,0.500956
3,0.573744,0.531599,0.510714,0.520947
4,0.593193,0.558140,0.512456,0.534323
5,0.581848,0.546939,0.476868,0.509506
6,0.554295,0.510563,0.516014,0.513274
7,0.565640,0.523810,0.508897,0.516245
8,0.598055,0.561798,0.533808,0.547445
9,0.570502,0.530769,0.491103,0.510166


## Tabla de predicciones para auditoría

`audit_predictions` contiene una fila por observación y utiliza el formato requerido por Aequitas.

Columnas principales:

- `entity_id`: identificador de la observación.
- `model_id`: nombre del modelo auditado.
- `feature_set`: estrategia de preprocesamiento utilizada.
- `fold`: partición que produjo la predicción.
- `race`, `sex`, `age_cat`: atributos protegidos originales.
- `label_value`: valor real de reincidencia.
- `score`: clase predicha por el modelo, 0 o 1.

En este proyecto, `score` representa una **decisión binaria**, no una probabilidad.

In [8]:
print("Número de predicciones para auditoría:", len(audit_predictions))
display(audit_predictions.head(10))

Número de predicciones para auditoría: 6172


,entity_id,model_id,feature_set,fold,race,sex,age_cat,label_value,score
0,3,LR_PrototypeRepresentationLearner,PrototypeRepresentationLearner,1,African-American,Male,25 - 45,1,0
1,14,LR_PrototypeRepresentationLearner,PrototypeRepresentationLearner,1,Caucasian,Male,25 - 45,0,1
2,42,LR_PrototypeRepresentationLearner,PrototypeRepresentationLearner,1,African-American,Male,25 - 45,1,0
3,45,LR_PrototypeRepresentationLearner,PrototypeRepresentationLearner,1,Caucasian,Male,Greater than 45,0,0
4,70,LR_PrototypeRepresentationLearner,PrototypeRepresentationLearner,1,African-American,Male,25 - 45,0,1
5,71,LR_PrototypeRepresentationLearner,PrototypeRepresentationLearner,1,Caucasian,Male,25 - 45,1,1
6,85,LR_PrototypeRepresentationLearner,PrototypeRepresentationLearner,1,Caucasian,Male,Greater than 45,0,0
7,100,LR_PrototypeRepresentationLearner,PrototypeRepresentationLearner,1,African-American,Female,Greater than 45,0,0
8,102,LR_PrototypeRepresentationLearner,PrototypeRepresentationLearner,1,Hispanic,Male,Less than 25,0,1
9,114,LR_PrototypeRepresentationLearner,PrototypeRepresentationLearner,1,Caucasian,Male,Less than 25,0,1


## Auditoría con Aequitas

La función `auditar_modelo` ejecuta tres niveles de análisis:

### Métricas absolutas por grupo

`Group.get_crosstabs` calcula métricas como:

- `prev`: prevalencia real de la clase positiva.
- `pprev`: proporción de predicciones positivas.
- `tpr`: tasa de verdaderos positivos.
- `tnr`: tasa de verdaderos negativos.
- `fpr`: tasa de falsos positivos.
- `fnr`: tasa de falsos negativos.
- `precision`: proporción de predicciones positivas correctas.

### Disparidades

`Bias.get_disparity_predefined_groups` divide la métrica de cada grupo entre la métrica del grupo de referencia.

Los grupos de referencia predeterminados son:

- `race`: `Caucasian`
- `sex`: `Male`
- `age_cat`: `25 - 45`

Una disparidad cercana a `1` indica comportamiento semejante al grupo de referencia. Una disparidad no demuestra por sí sola discriminación causal; señala diferencias que deben investigarse.

### Evaluación de equidad

Con `tau=0.80`, Aequitas aplica una regla de paridad equivalente al intervalo aproximado `[0.80, 1.25]`. Una razón fuera de ese intervalo se marca como una posible falla de paridad.

El parámetro `alpha=0.05` se utiliza para comprobar significancia estadística.

In [9]:
resultado = auditar_modelo(
    audit_predictions,
    tau=0.80,
    alpha=0.05,
)

## Selección de columnas relevantes

Aequitas produce tablas con muchas columnas. Para facilitar la lectura se crean vistas reducidas sin modificar los resultados completos guardados en `resultado`.

- La primera vista muestra métricas absolutas.
- La segunda muestra disparidades respecto al grupo de referencia.
- La tercera resume diferentes criterios de paridad.

In [10]:
ABSOLUTE_COLUMNS = [
    "model_id",
    "attribute_name",
    "attribute_value",
    "group_size",
    "prev",
    "pprev",
    "tpr",
    "tnr",
    "fpr",
    "fnr",
    "fdr",
    "for",
    "precision",
    "npv",
]

DISPARITY_COLUMNS = [
    "model_id",
    "attribute_name",
    "attribute_value",
    "group_size",
    "fpr_disparity",
    "fnr_disparity",
    "pprev_disparity",
    "fdr_disparity",
    "precision_disparity",
    "fpr_ref_group_value",
    "fnr_ref_group_value",
]

FAIRNESS_COLUMNS = [
    "model_id",
    "attribute_name",
    "attribute_value",
    "group_size",
    "FPR Parity",
    "FNR Parity",
    "Statistical Parity",
    "Impact Parity",
    "Equalized Odds",
    "TypeI Parity",
    "TypeII Parity",
    "Unsupervised Fairness",
    "Supervised Fairness",
]

group_metrics_view = resultado["group_metrics"].loc[:, ABSOLUTE_COLUMNS]
disparities_view = resultado["disparities"].loc[:, DISPARITY_COLUMNS]
group_fairness_view = resultado["group_fairness"].loc[:, FAIRNESS_COLUMNS]

## Resultados por grupo

Al interpretar estas tablas conviene revisar simultáneamente:

1. El tamaño del grupo (`group_size`), porque grupos pequeños producen estimaciones más inestables.
2. La métrica absoluta, para conocer la magnitud real del error.
3. La disparidad, para compararla con el grupo de referencia.
4. La significancia estadística, disponible en la tabla completa de disparidades.
5. El contexto del problema, ya que no existe una única métrica de equidad adecuada para todos los usos.

Por ejemplo, una diferencia en `fpr` indica que un grupo recibe falsos positivos con mayor frecuencia. Una diferencia en `fnr` indica que el modelo omite positivos reales con distinta frecuencia.

In [11]:
print("Métricas absolutas por grupo:")
display(group_metrics_view)

print("\nDisparidades respecto al grupo de referencia:")
display(disparities_view)

print("\nEvaluaciones de equidad por grupo:")
display(group_fairness_view)

Métricas absolutas por grupo:


,model_id,attribute_name,attribute_value,group_size,prev,pprev,tpr,tnr,fpr,fnr,fdr,for,precision,npv
0,LR_PrototypeRepresentationLearner,race,African-American,3175,0.523150,0.490394,0.544852,0.569353,0.430647,0.455148,0.418754,0.467244,0.581246,0.532756
1,LR_PrototypeRepresentationLearner,race,Asian,31,0.258065,0.322581,0.375000,0.695652,0.304348,0.625000,0.700000,0.238095,0.300000,0.761905
2,LR_PrototypeRepresentationLearner,race,Caucasian,2103,0.390870,0.391821,0.464720,0.654957,0.345043,0.535280,0.536408,0.344019,0.463592,0.655981
3,LR_PrototypeRepresentationLearner,race,Hispanic,509,0.371316,0.345776,0.428571,0.703125,0.296875,0.571429,0.539773,0.324324,0.460227,0.675676
4,LR_PrototypeRepresentationLearner,race,Native American,11,0.454545,0.181818,0.400000,1.000000,0.000000,0.600000,0.000000,0.333333,1.000000,0.666667
5,LR_PrototypeRepresentationLearner,race,Other,343,0.361516,0.067055,0.145161,0.977169,0.022831,0.854839,0.217391,0.331250,0.782609,0.668750
6,LR_PrototypeRepresentationLearner,sex,Female,1175,0.351489,0.406809,0.457627,0.620735,0.379265,0.542373,0.604603,0.321377,0.395397,0.678623
7,LR_PrototypeRepresentationLearner,sex,Male,4997,0.479488,0.423054,0.501669,0.649366,0.350634,0.498331,0.431410,0.414152,0.568590,0.585848
8,LR_PrototypeRepresentationLearner,age_cat,25 - 45,3532,0.464609,0.367214,0.400366,0.661555,0.338445,0.599634,0.493446,0.440268,0.506554,0.559732
9,LR_PrototypeRepresentationLearner,age_cat,Greater than 45,1293,0.320186,0.000000,0.000000,1.000000,0.000000,1.000000,NaN,0.320186,NaN,0.679814



Disparidades respecto al grupo de referencia:


,model_id,attribute_name,attribute_value,group_size,fpr_disparity,fnr_disparity,pprev_disparity,fdr_disparity,precision_disparity,fpr_ref_group_value,fnr_ref_group_value
0,LR_PrototypeRepresentationLearner,race,African-American,3175,1.248098,0.850298,1.251575,0.780664,1.253787,Caucasian,Caucasian
1,LR_PrototypeRepresentationLearner,race,Asian,31,0.882058,1.167614,0.823285,1.304977,0.647120,Caucasian,Caucasian
2,LR_PrototypeRepresentationLearner,race,Caucasian,2103,1.000000,1.000000,1.000000,1.000000,1.000000,Caucasian,Caucasian
3,LR_PrototypeRepresentationLearner,race,Hispanic,509,0.860400,1.067532,0.882484,1.006273,0.992742,Caucasian,Caucasian
4,LR_PrototypeRepresentationLearner,race,Native American,11,0.000000,1.120909,0.464034,0.000000,2.157068,Caucasian,Caucasian
5,LR_PrototypeRepresentationLearner,race,Other,343,0.066169,1.596994,0.171138,0.405272,1.688140,Caucasian,Caucasian
6,LR_PrototypeRepresentationLearner,sex,Female,1175,1.081654,1.088380,0.961600,1.401458,0.695400,Male,Male
7,LR_PrototypeRepresentationLearner,sex,Male,4997,1.000000,1.000000,1.000000,1.000000,1.000000,Male,Male
8,LR_PrototypeRepresentationLearner,age_cat,25 - 45,3532,1.000000,1.000000,1.000000,1.000000,1.000000,25 - 45,25 - 45
9,LR_PrototypeRepresentationLearner,age_cat,Greater than 45,1293,0.000000,1.667683,0.000000,NaN,NaN,25 - 45,25 - 45



Evaluaciones de equidad por grupo:


,model_id,attribute_name,attribute_value,group_size,FPR Parity,FNR Parity,Statistical Parity,Impact Parity,Equalized Odds,TypeI Parity,TypeII Parity,Unsupervised Fairness,Supervised Fairness
0,LR_PrototypeRepresentationLearner,race,African-American,3175,True,True,False,False,True,False,False,False,False
1,LR_PrototypeRepresentationLearner,race,Asian,31,True,True,False,True,True,False,False,False,False
2,LR_PrototypeRepresentationLearner,race,Caucasian,2103,True,True,True,True,True,True,True,True,True
3,LR_PrototypeRepresentationLearner,race,Hispanic,509,True,True,False,True,True,True,True,False,True
4,LR_PrototypeRepresentationLearner,race,Native American,11,False,True,False,False,False,False,True,False,False
5,LR_PrototypeRepresentationLearner,race,Other,343,False,False,False,False,False,False,False,False,False
6,LR_PrototypeRepresentationLearner,sex,Female,1175,True,True,False,True,True,False,False,False,False
7,LR_PrototypeRepresentationLearner,sex,Male,4997,True,True,True,True,True,True,True,True,True
8,LR_PrototypeRepresentationLearner,age_cat,25 - 45,3532,True,True,True,True,True,True,True,True,True
9,LR_PrototypeRepresentationLearner,age_cat,Greater than 45,1293,False,False,False,False,False,False,False,False,False


## Resúmenes por atributo y del modelo completo

`attribute_fairness` agrega los resultados de todos los valores de un atributo, por ejemplo todos los grupos de `race`.

`overall_fairness` resume la evaluación del modelo completo. Este resumen es útil para identificar rápidamente si existe alguna falla, pero no sustituye la inspección de las métricas y disparidades específicas.

In [ ]:
print("Resumen de equidad por atributo:")
display(resultado["attribute_fairness"])

print("\nResumen global de equidad:")
display(resultado["overall_fairness"])

# Guardar resultados en archivos CSV
output_dir = Path("results") / PREPROCESSING_METHOD
output_dir.mkdir(parents=True, exist_ok=True)

group_metrics_view.to_csv(output_dir / "group_metrics.csv", index=False)
disparities_view.to_csv(output_dir / "disparities.csv", index=False)
group_fairness_view.to_csv(output_dir / "group_fairness.csv", index=False

Resumen de equidad por atributo:


,model_id,score_threshold,attribute_name,Statistical Parity,Impact Parity,FDR Parity,FPR Parity,FOR Parity,FNR Parity,TPR Parity,TNR Parity,NPV Parity,Precision Parity,TypeI Parity,TypeII Parity,Equalized Odds,Unsupervised Fairness,Supervised Fairness
0,LR_PrototypeRepresentationLearner,binary 0/1,age_cat,False,False,True,False,False,False,False,False,True,True,False,False,False,False,False
1,LR_PrototypeRepresentationLearner,binary 0/1,race,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
2,LR_PrototypeRepresentationLearner,binary 0/1,sex,False,True,False,True,False,True,True,True,True,False,False,False,True,False,False



Resumen global de equidad:


{'Unsupervised Fairness': False,
 'Supervised Fairness': False,
 'Overall Fairness': False}

## 11. Cómo comparar estrategias

Para evaluar el efecto de Fairlearn, se recomienda ejecutar el notebook con:

1. `PREPROCESSING_METHOD = None`
2. `PREPROCESSING_METHOD = "CorrelationRemover"`
3. `PREPROCESSING_METHOD = "PrototypeRepresentationLearner"`

Después se deben comparar dos dimensiones:

- **Utilidad predictiva**: cambios en `accuracy`, `precision`, `recall` y `F1`.
- **Equidad**: cambios en `fpr_disparity`, `fnr_disparity`, `pprev_disparity` y los indicadores de paridad.

Una técnica de mitigación no garantiza que todas las métricas mejoren simultáneamente. Es normal encontrar compromisos entre rendimiento, distintos criterios de equidad y grupos diferentes.

## Limitaciones del análisis

- Las métricas describen asociaciones observadas; no prueban causalidad.
- Los resultados dependen de la calidad de las etiquetas y del proceso que generó los datos.
- Los grupos pequeños pueden producir métricas variables o poco confiables.
- La selección del grupo de referencia y del umbral `tau` afecta la evaluación.
- La equidad técnica debe complementarse con análisis jurídico, social y del contexto de uso.

## Trabajo futuro

- Implementar soporte para más metodos de mitigación de Fairlearn.